# Evaluating Performance

In [3]:
# Importing packages
import pandas as pd
import torch
import os
os.environ["HF_HUB_DISABLE_PROGREvSS_BARS"] = "1"
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, logging, AutoModel
logging.set_verbosity_error()
import numpy as np
from concurrent.futures import ThreadPoolExecutor
from dotenv import load_dotenv
import os
from torch import Tensor
import faiss 
import json
from beir.datasets.data_loader import GenericDataLoader
import torch.nn.functional as F
from beir.retrieval.evaluation import EvaluateRetrieval

In [4]:
### Loading Data ###
data_dir = "/work/mbouthil/datasets/msmarco"
corpus, dev_queries, dev_qrels = GenericDataLoader(data_folder=data_dir).load(split="dev")

100%|██████████| 8841823/8841823 [01:16<00:00, 115448.08it/s]


In [5]:
print(len(dev_queries))

6980


In [6]:
dev_info = [(key, value) for key, value in dev_queries.items()]

In [7]:
# Loading Index
index = faiss.read_index("/work/mbouthil/MMATH-CM-Research-Project/RAG/retrieval_data/passage_v01.index")

In [8]:
# Loading Query Encoder
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
query_encoder = AutoModel.from_pretrained(
    "/work/mbouthil/MMATH-CM-Research-Project/RAG/model_weights/query_encoder_v01"
).to(device)
query_encoder.eval()

### PROBEM IS HERE


def encode_query(query:str) -> Tensor:

    queries = [query] if isinstance(query, str) else query
    embeddings = []

    with torch.no_grad():
        inputs = tokenizer(
            queries, 
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=32
        ).to(device)

    out = query_encoder(**inputs)
    last_hidden_state = out.last_hidden_state
    mask = inputs['attention_mask'].unsqueeze(-1).float()
    emb = (last_hidden_state * mask).sum(dim=1) / mask.sum(dim=1)

    emb = F.normalize(emb, p=2, dim=-1)
    embeddings.append(emb.cpu())

    return torch.cat(embeddings, 0)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 306.62it/s, Materializing param=pooler.dense.weight]                               


In [9]:
# Loading passage Encoder
passage_encoder = AutoModel.from_pretrained(
    "/work/mbouthil/MMATH-CM-Research-Project/RAG/model_weights/passage_encoder_syn_que_0"
).to(device)
passage_encoder.eval()

def encode_passage(passage:str) -> Tensor:

    passages = [passage] if isinstance(passage, str) else passage
    embeddings = []

    with torch.no_grad():
        inputs = tokenizer(
            passages, 
            padding=True,
            truncation=True,
            return_tensors="pt",
            max_length=128
        ).to(device)

    emb = passage_encoder(**inputs).last_hidden_state[:, 0]
    emb = F.normalize(emb, p=2, dim=-1)

    embeddings.append(emb.cpu())

    return torch.cat(embeddings, 0)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 205.45it/s, Materializing param=pooler.dense.weight]                               


In [10]:
def batch_splits(item:list, batch_size:int=50):

    for i in range(0, len(item), batch_size):
        yield item[i:i + batch_size]

batches = batch_splits(dev_info)

In [15]:
answers = {}

for batch in batches:

    q_ids, queries = zip(*batch)
    N = range(len(q_ids))
    
    q_emb = encode_query(queries).detach().cpu().numpy()
    scores, pids = index.search(q_emb, 100)
    
    pids = [[str(pid) for pid in pids[i]] for i in N]
    scores = [[float(score) for score in scores[i]] for i in N]
    
    batch_results = {
        q_ids[i]: dict(zip(pids[i], scores[i])) 
        for i in N
    }
    
    answers.update(batch_results)
    break

In [16]:
results = answers
print(results)

{'127876': {'1999615': 0.9975860714912415, '7075541': 0.9975095987319946, '1999616': 0.9973831176757812, '8605398': 0.9972736239433289, '1845655': 0.997240424156189, '780008': 0.9972305297851562, '1845646': 0.9971238970756531, '5428734': 0.9970957040786743, '3329700': 0.9970647096633911, '7653033': 0.9970430731773376, '3247865': 0.9970331788063049, '8457939': 0.9969937801361084, '2963051': 0.9969846606254578, '204578': 0.996959924697876, '776671': 0.9969214200973511, '2962940': 0.9969093799591064, '5239056': 0.9968929290771484, '8605397': 0.9968512058258057, '204584': 0.9968305230140686, '204577': 0.9968127012252808, '7891610': 0.9968104362487793, '8137109': 0.9967480897903442, '6931587': 0.9967395663261414, '289398': 0.9966992139816284, '8448680': 0.9966953992843628, '1939878': 0.9966893196105957, '1581300': 0.996654212474823, '204579': 0.9966431856155396, '1359001': 0.9966369867324829, '204583': 0.9966306686401367, '204582': 0.9966216683387756, '5760028': 0.9966205954551697, '2963057

In [17]:
ndcg, _map, recall, precision = EvaluateRetrieval.evaluate(
    dev_qrels,
    results, 
    k_values=[1, 3, 5, 10, 100, 1000]
)

In [19]:
print(ndcg['NDCG@10'])
print(recall['Recall@100'])

0.08
0.14


In [ ]:
mrr = EvaluateRetrieval.evaluate_custom(
    dev_qrels, 
    results, 
    k_values=[100], 
    metric="mrr"
)

print(f"MRR@10: {mrr['MRR@100']}") 

MRR@10: 0.00021


In [29]:
print(2e-5)

2e-05


In [30]:
1*200000/60/60

55.55555555555556

In [39]:
def recall_at_k(results, qrels, k):
    hits = 0
    total = 0

    for qid, rels in qrels.items():
        if qid not in results.keys():
            continue

        retrieved_pids = list(results[qid].keys())[:k]
        relevant_pids = set(rels.keys())

        if len(set(retrieved_pids) & relevant_pids) > 0:
            hits += 1

        total += 1

    return hits / total

In [41]:
recall_at_k(results, dev_qrels, 100)

0.22

In [42]:
import math

def ndcg_at_10(results, qrels):
    total_ndcg = 0.0
    count = 0

    for qid, rels in qrels.items():
        if qid not in results:
            continue

        retrieved_pids = list(results[qid].keys())[:10]
        relevant_pids = set(rels.keys())

        dcg = 0.0
        for i, pid in enumerate(retrieved_pids):
            if pid in relevant_pids:
                dcg += 1.0 / math.log2(i + 2)  # i+1 rank, +1 for log

        # For MS MARCO: one relevant doc ⇒ IDCG = 1
        idcg = 1.0

        total_ndcg += dcg / idcg
        count += 1

    return total_ndcg / count


In [44]:
ndcg_at_10(results, dev_qrels)

0.03125779005613793